## What is LangGraph?

LangGraph is an open-source framework and orchestration engine created by LangChain for building stateful, multi-actor AI agents. 

It allow to define AI workflows using a graph-based architecture—where nodes represent specific tasks (like calling an LLM or running a tool) and edges manage complex logic like loops, conditional branching, and parallel processing. 

## when to use LangGraph vs LangChain

LangChain is used to build LLM applications with relatively straightforward workflows, such as chatbots, RAG systems, and prompt chains. It works well when the execution flow is mostly linear.

LangGraph is used when building advanced AI agents that require state management, conditional branching, loops, retries, multi-agent collaboration, human approvals, or long-running workflows. In fact, LangGraph is built on top of LangChain and extends it with graph-based orchestration.

## Use cases
Here are the **real-world use cases of LangGraph** in clear, concise sentences:

1. **Research Agents:** Build AI agents that can search for information, evaluate results, and iterate until they find the best answer.
2. **Code Review Agents:** Create agents that analyze code, suggest improvements, recheck changes, and validate the final output.
3. **Approval Workflows:** Design workflows that include human approvals and reviews, even if the process spans multiple days.
4. **AI and Multi-Agent Systems:** Develop intelligent AI agents that collaborate with other agents to solve complex tasks.
5. **Long-Running Workflows:** Manage stateful processes that require multiple steps, decision-making, retries, and continuous execution over time.

## Key Concepts

## State: 

A shared data structure (like a whiteboard) that acts as the application's memory. Every node can read from it and write updates back to it as the workflow runs.

```python
from typing import TypedDict

class State(TypedDict):
    question: str
    answer: str
```

## Nodes: 

Plain Python or TypeScript functions that perform specific operations. They take the current state, process it, and return updated values. 

```python
def chatbot(state: State):
    return {
        "answer": "Hello!"
    }
```

## Edges:

Connections that determine the flow of execution. They can be static or conditional (e.g., "if the data is invalid, return to the editing node")

## why handoffs in LangGprah

Handoffs in LangGraph allow one agent to transfer control to another agent when a different agent is better suited to handle the next part of the task.

### Why are handoffs needed?
- Specialized agents – Different agents can have different expertise (e.g., a Research Agent, Coding Agent, and Review Agent).
- Task delegation – An agent can pass a task to another agent that is better equipped to complete it.
- Modular workflows – Each agent focuses on a specific responsibility, making the system easier to build and maintain.
- Improved accuracy – The most appropriate agent handles each step, leading to better results.
- Scalable multi-agent systems – New agents can be added without redesigning the entire workflow.

## Direct Edge VS Conditional Edge

### Direct Edge: 

A direct edge connects one node to another in a fixed sequence. After the current node finishes execution, the graph always moves to the same next node without checking any conditions. It is best for simple, linear workflows.

```python
graph.add_edge("A", "B")
```

### Conditional Edge: 

A conditional edge evaluates the current state or the output of a node to decide which node should execute next. This enables branching, routing, retries, tool selection, and other dynamic behaviors, making it ideal for AI agents and complex workflows.

```python
graph.add_conditional_edges("A", router, {
    "search": "search",
    "answer": "answer"
})
```

## Basic Routing VS Literal Routing 

### Basic Routing

Basic routing uses a routing function with standard strings to determine the next node. The function contains custom logic (such as if-else conditions) to decide where the graph should go next.

```python
def router(state):
    if state["need_search"]:
        return "search"
    return "generate"
```

### Literal Routing

Literal routing uses Python's Literal type to restrict the router's possible outputs to a predefined set of values. This makes the routing logic more type-safe, easier to validate, and improves IDE support.

```python
from typing import Literal

def router(state) -> Literal["search", "generate"]:
    if state["need_search"]:
        return "search"
    return "generate"
```

## Multipath Routing

**Multipath routing** allows a node to route execution to **multiple next nodes simultaneously** instead of just one. It is useful when multiple tasks can run independently in parallel.

### Example

```python
def router(state):
    return ["search", "calculator"]

graph.add_conditional_edges(
    "planner",
    router,
    {
        "search": "search",
        "calculator": "calculator"
    }
)


## Cycles and Loops in LangGraph

A cycle (loop) in LangGraph allows the graph to return to a previous node and repeat execution until a condition is met. This enables iterative workflows where an agent can retry, refine, or continue working until it achieves the desired result.

### 1. Self-Correcting Agent

A **self-correcting agent** generates an output, evaluates whether it is correct, and if it finds errors, it **loops back to fix its own mistakes**. The process repeats until the output meets the required quality.

### 2. Iterative Agent

An **iterative agent** repeats a task to **collect more information or improve the result**, rather than fixing mistakes.

It continues until it reaches a stopping condition.


## Human-in-the-Loop (HITL)

A Human-in-the-Loop (HITL) workflow allows a human to review, approve, reject, or modify an AI agent's output before the graph continues execution. It combines AI automation with human oversight for critical decisions.


### interrupt_before

Pauses the graph before a specified node executes. This allows a human to review or approve the action before it happens.

Use Case: Approve sending an email,
Approve a payment 
Review a generated SQL query before execution.

```python
graph = builder.compile(
    checkpointer=MemorySaver(),
    interrupt_before=["send_email"]
)
```

interrupt_after

Pauses the graph after a specified node has executed. This allows a human to review the output before the workflow continues.

Use Case: Review AI-generated content, Verify a generated report, Check code generated by an AI agent.

```python
graph = builder.compile(
    checkpointer=MemorySaver(),
    interrupt_after=["generate_report"]
)


## Checkpointer in LangGraph

A Checkpointer is a persistence mechanism that saves the graph's state after each step. This allows the workflow to pause, resume, recover from failures, and support Human-in-the-Loop interactions.


### MemorySaver 

is an in-memory checkpointer that stores the graph's execution state in RAM. It enables workflows to pause and resume during the application's lifetime, but all saved state is lost when the application stops or restarts.

Best for: Development, Testing, Temporary workflows

```python
from langgraph.graph import StateGraph
from langgraph.checkpoint.memory import MemorySaver

memory = MemorySaver()

graph = builder.compile(
    checkpointer=memory
)
```

### SqliteSaver 

is a persistent checkpointer that stores the graph's execution state in a SQLite database. Since the state is saved on disk, workflows can resume even after the application or server restarts.

Best for:Production applications, Human-in-the-Loop workflows, Long-running agents, Fault recovery

```python
from langgraph.checkpoint.sqlite import SqliteSaver

checkpointer = SqliteSaver.from_conn_string("checkpoints.db")

graph = builder.compile(
    checkpointer=checkpointer
)
```

# A Reducer in LangGraph 

is a function that defines how updates to the same state field are combined when multiple nodes write to it. Reducers are especially important in parallel workflows to prevent data loss and ensure all relevant outputs are merged correctly.

```python 

from typing import Annotated
from typing_extensions import TypedDict
from operator import add

class State(TypedDict):
    messages: Annotated[list, add]

def node1(state):
    return {
        "messages": ["Hello"]
    }

def node2(state):
    return {
        "messages": ["How are you?"]
```

## MessagesState in LangGraph

MessagesState is a built-in state class in LangGraph that automatically stores and manages the conversation history (messages) between the user, AI, and tools.

Instead of creating your own messages field, you can use MessagesState.

### Key Concepts
- Built-in state for chat applications.
- Automatically stores conversation history.
- Maintains messages across nodes.
- Compatible with LangChain message types (HumanMessage, AIMessage, ToolMessage, etc.).
- Simplifies chatbot and agent development.

```python 

from langgraph.graph import MessagesState

class State(MessagesState):
    pass

from langchain_core.messages import AIMessage

def chatbot(state: MessagesState):
    return {
        "messages": [
            AIMessage(content="Hello! How can I help you?")
        ]
    }
```